In [5]:
import os
import glob
import numpy as np
import pandas as pd

# ===== 改成你本地的 results 根目录 =====
ROOT_DIR = r"C:\Users\yangy\two_split\results"   # 注意路径里的空格，用 r""

# 文件模式：rep_* 里的 pointwise_coverage_repXXX_A*_B*.csv
pattern = os.path.join(ROOT_DIR, "rep_*", "pointwise_coverage_rep*_A*_B*.csv")
file_list = sorted(glob.glob(pattern))

print(f"找到 {len(file_list)} 个 csv 文件")
assert len(file_list) > 0, "没有匹配到文件，检查 ROOT_DIR 和 pattern"

# 如果你确实是 100 个 rep，可以再加一层保险：
# assert len(file_list) == 100, "不是 100 个文件，再看看哪里不对"

# ===== 读第一个文件，拿到 x 和 alpha 列名 =====
first_df = pd.read_csv(file_list[0])
x_vals = first_df["x"].values
alpha_cols = [c for c in first_df.columns if c != "x"]
# 列名形如 alpha_0.010 -> 拿出数值
alphas = [float(c.split("_")[1]) for c in alpha_cols]
A = len(alpha_cols)
J = len(x_vals)
print(f"x 网格点数 J={J}, alpha 个数 A={A}")

# ===== 把所有 rep 的表堆成 [R, J, A] =====
values_list = []
for fname in file_list:
    df = pd.read_csv(fname)
    # sanity check: x 是否一致
    if not np.allclose(df["x"].values, x_vals):
        raise ValueError(f"x 网格在文件 {os.path.basename(fname)} 中不一致")
    vals = df[alpha_cols].values  # [J, A]
    values_list.append(vals)

arr = np.stack(values_list, axis=0)   # [R, J, A]
R = arr.shape[0]
print(f"数组形状: R={R}, J={J}, A={A}")

找到 100 个 csv 文件
x 网格点数 J=50, alpha 个数 A=100
数组形状: R=100, J=50, A=100


In [6]:
# 对 R 取平均 -> [J, A]
mean_cov = arr.mean(axis=0)
std_cov  = arr.std(axis=0)   # 可选：看波动大小

# 生成 DataFrame：x + 每个 alpha 的平均 coverage
data_mean = {"x": x_vals}
data_std  = {"x": x_vals}
for a_idx, col in enumerate(alpha_cols):
    data_mean[col] = mean_cov[:, a_idx]
    data_std[col]  = std_cov[:, a_idx]

df_mean = pd.DataFrame(data_mean)
df_std  = pd.DataFrame(data_std)

display(df_mean.head())
display(df_std.head())

# 保存到 ROOT_DIR
out_mean = os.path.join(ROOT_DIR, "combined_pointwise_coverage_mean_over_reps.csv")
out_std  = os.path.join(ROOT_DIR, "combined_pointwise_coverage_std_over_reps.csv")
df_mean.to_csv(out_mean, index=False)
df_std.to_csv(out_std, index=False)

print("保存完毕：")
print("  平均覆盖 ->", out_mean)
print("  覆盖标准差 ->", out_std)


,x,alpha_0.0100,alpha_0.0200,alpha_0.0300,alpha_0.0400,alpha_0.0500,alpha_0.0600,alpha_0.0700,alpha_0.0800,alpha_0.0900,...,alpha_0.9100,alpha_0.9200,alpha_0.9300,alpha_0.9400,alpha_0.9500,alpha_0.9600,alpha_0.9700,alpha_0.9800,alpha_0.9900,alpha_1.0000
0,-2.500000,1.0,1.0,1.0,0.99985,0.9990,0.99735,0.99235,0.98860,0.98195,...,0.7600,0.75860,0.75195,0.75185,0.75150,0.75070,0.74465,0.74665,0.74515,0.74020
1,-2.397959,1.0,1.0,1.0,1.00000,1.0000,1.00000,1.00000,1.00000,1.00000,...,0.9673,0.96745,0.96840,0.96570,0.96630,0.96435,0.96540,0.96565,0.96410,0.96370
2,-2.295918,1.0,1.0,1.0,1.00000,1.0000,1.00000,1.00000,0.99995,0.99980,...,0.8353,0.83380,0.83150,0.82990,0.83055,0.82820,0.82605,0.82405,0.82605,0.82145
3,-2.193878,1.0,1.0,1.0,0.99995,0.9985,0.99490,0.99205,0.99025,0.98895,...,0.7416,0.74020,0.73785,0.73620,0.73355,0.73040,0.72770,0.72570,0.72570,0.72095
4,-2.091837,1.0,1.0,1.0,1.00000,1.0000,1.00000,1.00000,1.00000,1.00000,...,0.9374,0.93710,0.93735,0.93515,0.93570,0.93435,0.93365,0.93305,0.93205,0.93005


,x,alpha_0.0100,alpha_0.0200,alpha_0.0300,alpha_0.0400,alpha_0.0500,alpha_0.0600,alpha_0.0700,alpha_0.0800,alpha_0.0900,...,alpha_0.9100,alpha_0.9200,alpha_0.9300,alpha_0.9400,alpha_0.9500,alpha_0.9600,alpha_0.9700,alpha_0.9800,alpha_0.9900,alpha_1.0000
0,-2.500000,0.0,0.0,0.0,0.001492,0.007141,0.016178,0.042122,0.057027,0.083081,...,0.365818,0.366395,0.368551,0.369611,0.370543,0.368995,0.372362,0.369287,0.368570,0.371589
1,-2.397959,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.147072,0.146831,0.145317,0.151672,0.151120,0.155149,0.149896,0.151891,0.156029,0.157117
2,-2.295918,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000497,0.001568,...,0.334687,0.336297,0.337370,0.342404,0.340458,0.342231,0.347182,0.348102,0.344857,0.348183
3,-2.193878,0.0,0.0,0.0,0.000497,0.014925,0.050744,0.078601,0.093526,0.096918,...,0.399837,0.402107,0.402117,0.401675,0.403017,0.404582,0.407457,0.407855,0.406800,0.410308
4,-2.091837,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.223382,0.223618,0.223523,0.226001,0.225628,0.227221,0.227881,0.228457,0.228734,0.233611


保存完毕：
  平均覆盖 -> C:\Users\yangy\two_split\results\combined_pointwise_coverage_mean_over_reps.csv
  覆盖标准差 -> C:\Users\yangy\two_split\results\combined_pointwise_coverage_std_over_reps.csv


In [1]:
import os
import glob
import numpy as np
import pandas as pd

# ===== 改成你本地的 results 根目录 =====
ROOT_DIR = r"C:\Users\yangy\two_split\results"

# 文件模式：rep_* 里的 pointwise_coverage_repXXX_A*_B*.csv
pattern = os.path.join(ROOT_DIR, "rep_*", "pointwise_coverage_rep*_A*_B*.csv")
file_list = sorted(glob.glob(pattern))

print(f"找到 {len(file_list)} 个 csv 文件")
assert len(file_list) > 0, "没有匹配到文件，检查 ROOT_DIR 和 pattern"
# 如果你期望正好 100 个 rep，可以开这个
# assert len(file_list) == 100, "不是 100 个文件，再看看哪里不对"

# ===== 读第一个文件，拿到 x 和 alpha 列名 =====
first_df = pd.read_csv(file_list[0])
x_vals = first_df["x"].values
alpha_cols = [c for c in first_df.columns if c != "x"]
# 或者更安全一点：startswith("alpha_")
# alpha_cols = [c for c in first_df.columns if c.startswith("alpha_")]

alphas = [float(c.split("_")[1]) for c in alpha_cols]
A = len(alpha_cols)
J = len(x_vals)
print(f"x 网格点数 J={J}, alpha 个数 A={A}")

# ===== 把所有 rep 的表堆成 [R, J, A] =====
values_list = []
for fname in file_list:
    df = pd.read_csv(fname)
    # sanity check: x 是否一致
    if not np.allclose(df["x"].values, x_vals):
        raise ValueError(f"x 网格在文件 {os.path.basename(fname)} 中不一致")
    vals = df[alpha_cols].values  # [J, A]
    values_list.append(vals)

arr = np.stack(values_list, axis=0)   # [R, J, A]
R = arr.shape[0]
print(f"数组形状: R={R}, J={J}, A={A}")

# ===== 对 R 取平均 / std -> [J, A] =====
mean_cov = arr.mean(axis=0)
std_cov  = arr.std(axis=0)   # 看 rep 间波动

# ===== 生成 DataFrame：x + 每个 alpha 的平均 coverage =====
data_mean = {"x": x_vals}
data_std  = {"x": x_vals}
for a_idx, col in enumerate(alpha_cols):
    data_mean[col] = mean_cov[:, a_idx]
    data_std[col]  = std_cov[:, a_idx]

df_mean = pd.DataFrame(data_mean)
df_std  = pd.DataFrame(data_std)

display(df_mean.head())
display(df_std.head())

# ===== 保存到 ROOT_DIR =====
out_mean = os.path.join(ROOT_DIR, "combined_mean.csv")
out_std  = os.path.join(ROOT_DIR, "combined_std.csv")
df_mean.to_csv(out_mean, index=False)
df_std.to_csv(out_std, index=False)

print("保存完毕：")
print("  平均覆盖 ->", out_mean)
print("  覆盖标准差 ->", out_std)


找到 100 个 csv 文件
x 网格点数 J=50, alpha 个数 A=100
数组形状: R=100, J=50, A=100


,x,alpha_0.0100,alpha_0.0200,alpha_0.0300,alpha_0.0400,alpha_0.0500,alpha_0.0600,alpha_0.0700,alpha_0.0800,alpha_0.0900,...,alpha_0.9100,alpha_0.9200,alpha_0.9300,alpha_0.9400,alpha_0.9500,alpha_0.9600,alpha_0.9700,alpha_0.9800,alpha_0.9900,alpha_1.0000
0,-2.500000,1.0,1.0,1.0,0.99985,0.9990,0.99735,0.99235,0.98860,0.98195,...,0.7600,0.75860,0.75195,0.75185,0.75150,0.75070,0.74465,0.74665,0.74515,0.74020
1,-2.397959,1.0,1.0,1.0,1.00000,1.0000,1.00000,1.00000,1.00000,1.00000,...,0.9673,0.96745,0.96840,0.96570,0.96630,0.96435,0.96540,0.96565,0.96410,0.96370
2,-2.295918,1.0,1.0,1.0,1.00000,1.0000,1.00000,1.00000,0.99995,0.99980,...,0.8353,0.83380,0.83150,0.82990,0.83055,0.82820,0.82605,0.82405,0.82605,0.82145
3,-2.193878,1.0,1.0,1.0,0.99995,0.9985,0.99490,0.99205,0.99025,0.98895,...,0.7416,0.74020,0.73785,0.73620,0.73355,0.73040,0.72770,0.72570,0.72570,0.72095
4,-2.091837,1.0,1.0,1.0,1.00000,1.0000,1.00000,1.00000,1.00000,1.00000,...,0.9374,0.93710,0.93735,0.93515,0.93570,0.93435,0.93365,0.93305,0.93205,0.93005


,x,alpha_0.0100,alpha_0.0200,alpha_0.0300,alpha_0.0400,alpha_0.0500,alpha_0.0600,alpha_0.0700,alpha_0.0800,alpha_0.0900,...,alpha_0.9100,alpha_0.9200,alpha_0.9300,alpha_0.9400,alpha_0.9500,alpha_0.9600,alpha_0.9700,alpha_0.9800,alpha_0.9900,alpha_1.0000
0,-2.500000,0.0,0.0,0.0,0.001492,0.007141,0.016178,0.042122,0.057027,0.083081,...,0.365818,0.366395,0.368551,0.369611,0.370543,0.368995,0.372362,0.369287,0.368570,0.371589
1,-2.397959,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.147072,0.146831,0.145317,0.151672,0.151120,0.155149,0.149896,0.151891,0.156029,0.157117
2,-2.295918,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000497,0.001568,...,0.334687,0.336297,0.337370,0.342404,0.340458,0.342231,0.347182,0.348102,0.344857,0.348183
3,-2.193878,0.0,0.0,0.0,0.000497,0.014925,0.050744,0.078601,0.093526,0.096918,...,0.399837,0.402107,0.402117,0.401675,0.403017,0.404582,0.407457,0.407855,0.406800,0.410308
4,-2.091837,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.223382,0.223618,0.223523,0.226001,0.225628,0.227221,0.227881,0.228457,0.228734,0.233611


保存完毕：
  平均覆盖 -> C:\Users\yangy\two_split\results\combined_mean.csv
  覆盖标准差 -> C:\Users\yangy\two_split\results\combined_std.csv
